# **Import Library**

In [ ]:
!pip install ydata-profiling

In [ ]:
# general
import pandas as pd
import numpy as np
import math
from scipy import stats

# visualization
from ydata_profiling import ProfileReport
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

# Feature Selection
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

# Oversampling
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

from sklearn.model_selection import cross_val_score

# Tuning
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV


# **Load Dataset**

In [ ]:
df = pd.read_csv('employee_churn_prediction_updated.csv')
df.head()

In [ ]:
df.info()

# **Exploratory Data Analysis (EDA)**

## **Describe Data**

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

## **Define Columns**

In [ ]:
# memisahkan kolom numerik dan kategorikal
numerical_columns = df.select_dtypes(include=[np.number]).columns
numerical_columns = numerical_columns.drop(['employee_id', 'churn'])

categorical_columns = df.select_dtypes(include=['object']).columns

print("Numerical Columns:")
print(numerical_columns)
print("\nCategorical Columns:")
print(categorical_columns)

In [ ]:
# The 'categorical_columns' variable was defined before the 'churn_period' column was dropped from df.
# Therefore, it contains 'churn_period' which is no longer in df, leading to a KeyError.
# We need to redefine 'categorical_columns' based on the current state of df.
categorical_columns = df.select_dtypes(include=['object']).columns

# Now, to get unique values for each categorical column, iterate through them.
for col in categorical_columns:
    print(f"Unique values for {col}: {df[col].unique()}")

## **Automatic EDA**

In [ ]:
profile = ProfileReport(df)

profile.to_file("auto_eda.html")

## **Check Missing Value & Duplicate Data**

In [ ]:
# Missing Value
missing_values = df.isna().sum()
columns_with_missing_values = missing_values[missing_values > 0]

if not columns_with_missing_values.empty:
    print("Columns with missing values and their counts:")
    print(columns_with_missing_values)
else:
    print("No columns with missing values found.")

In [ ]:
# Duplicate Data
duplicate_rows = df[df.duplicated()]

if not duplicate_rows.empty:
    print("Duplicate rows found:")
    print(duplicate_rows)
    print("Total duplicate rows:", len(duplicate_rows))
else:
    print("No duplicate rows found.")

## **Set Visualization**

In [ ]:
# 1. Pilih palette (contoh: 'viridis', 'magma', 'Blues_d', atau custom hex list)
# 'viridis' adalah palette default yang Anda gunakan di fungsi plot_features_by_churn
my_palette = sns.color_palette("bright")

# 2. Set secara global untuk Seaborn
sns.set_palette(my_palette)

# 3. Set style agar grafik terlihat lebih modern dan bersih
sns.set_style("white")

# 4. Set parameter global untuk Matplotlib (agar plt.plot juga ikut berubah)
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=my_palette)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

## **Univariate Analysis**

### **Outlier**

In [ ]:
plt.figure(figsize=(15, 18)) # Adjusted figure size to better accommodate all plots

num_plots = len(numerical_columns)
num_cols = 4  # Number of columns in the subplot grid
num_rows = math.ceil(num_plots / num_cols) # Calculate number of rows needed

for i, column in enumerate(numerical_columns):
    plt.subplot(num_rows, num_cols, i + 1)
    sns.boxplot(x=df[column])
    plt.title(f'Box Plot of {column}')

plt.tight_layout()
plt.show()

### **Numeric Distributions**

In [ ]:
num_plots = len(numerical_columns)
num_cols = 4
num_rows = math.ceil(num_plots / num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(20, 4 * num_rows))

# Flatten the axes array for easy iteration if num_rows > 1
axes = axes.flatten()

for i, col in enumerate(numerical_columns):
  ax = axes[i] # Get the current axis
  for label, grp in df.groupby('churn'):
    # Ensure there's enough data for KDE, otherwise skip or handle differently
    if len(grp[col].dropna().unique()) > 1:
        grp[col].plot.kde(
            ax=ax,
            label=['Churn', 'No Churn'][label], # Changed 'labels' to 'label'
            linewidth=2
        )
    else:
        # For columns with constant values in a group, a simple line might be more appropriate
        # Or, just skip if a KDE is impossible
        if not grp[col].empty:
            ax.axvline(x=grp[col].iloc[0], color=my_palette[label], linestyle='--', label=f"{['Churn', 'No Churn'][label]} (Constant)")

  ax.set_title(f'Distribution of {col}')
  ax.set_xlabel(col)
  ax.legend()

# Hide any unused subplots if the grid has more slots than plots
for j in range(len(numerical_columns), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    'Numeric Feature Distribution by Churn',
    fontsize=14,
    fontweight='bold'
)
plt.tight_layout(
    rect=[0, 0.03, 1, 0.95],
    h_pad=2.0,
    w_pad=2.0
) # Adjust rect for suptitle to prevent overlap
plt.show()

### **Numeric Distributions (Histograms)**

In [ ]:
num_plots = len(numerical_columns)
num_cols = 4
num_rows = math.ceil(num_plots / num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(20, 4 * num_rows))

# Flatten the axes array for easy iteration if num_rows > 1
axes = axes.flatten()

for i, col in enumerate(numerical_columns):
  ax = axes[i] # Get the current axis
  sns.histplot(
      data=df,
      x=col,
      hue='churn',
      kde=False, # Disable KDE for histogram
      palette=my_palette,
      ax=ax
  )
  ax.set_title(f'Distribution of {col}')
  ax.set_xlabel(col)
  ax.legend(title='Churn', labels=['Churn', 'No Churn'])

# Hide any unused subplots if the grid has more slots than plots
for j in range(len(numerical_columns), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    'Numeric Feature Distribution by Churn (Histograms)',
    fontsize=14,
    fontweight='bold'
)
plt.tight_layout(
    rect=[0, 0.03, 1, 0.95],
    h_pad=2.0,
    w_pad=2.0
)
plt.show()

### **Categorical Distibrution (Histogram)**

In [ ]:
def plot_features_by_churn(df, features, plot_type_mapping, num_cols=3, title_prefix="Distribution of"):
    num_rows = math.ceil(len(features) / num_cols)

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(5 * num_cols, 4 * num_rows))
    axes = axes.flatten()

    total_employees = len(df)

    for i, col in enumerate(features):
        ax = axes[i] # Assign the current subplot axis
        # The original code only contained a countplot, so I'll keep that for now.
        # The plot_type_mapping parameter is not utilized in this corrected version
        sns.countplot(data=df, x=col, hue='churn', ax=ax)
        ax.tick_params(axis='x')

        for container in ax.containers:
            for p in container.patches:
                height = p.get_height()
                if height > 0:
                    percentage = (height / total_employees) * 100
                    ax.annotate(f'{int(height)}\n({percentage:.1f}%)',
                                (p.get_x() + p.get_width() / 2., height),
                                ha='center', va='top', fontsize=7, color='white', xytext=(0, -5),
                                textcoords='offset points')

        ax.set_title(f'{col} by Churn Status')
        ax.legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])
        ax.set_xlabel(col)
        ax.set_ylabel('Count / Density')

    # Hide any unused subplots if the grid has more slots than plots
    for j in range(len(features), len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout(
    rect=[0, 0.03, 1, 0.95],
    h_pad=2.0,
    w_pad=2.0
    )
    plt.show()

In [ ]:
# Identify categorical features
categorical_features = df.select_dtypes(include='object').columns.tolist()

# Define plot types for categorical features (all countplot for this section)
categorical_plot_type_mapping = {feature: 'countplot' for feature in categorical_features}

# Plot categorical features
plot_features_by_churn(df, categorical_features, categorical_plot_type_mapping, num_cols=2, title_prefix="Categorical")

### **Churn**

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
churn_counts = df['churn'].value_counts()

colors = [sns.color_palette()[1], sns.color_palette()[0]]

ax.pie(
    churn_counts,
    labels=['(1) Churn', '(0) Stayed'],
    autopct='%1.1f%%',
    startangle=90,
    explode=[0, 0.05],
    colors=colors,
    # textprops={'fontsize': 10}
)
ax.set_title(
    'Churn Distribution',
    fontsize=12,
    fontweight='bold'
)
plt.tight_layout()
plt.show()

### **Churn Periode**

In [ ]:
pd.crosstab(df['churn_period'], df['churn'])

In [ ]:
df.groupby(['churn_period'])['company_tenure_years'].agg(['min', 'max', 'mean', pd.Series.mode])

In [ ]:
onboarding_df = df[df['churn_period'] == 'Onboarding']
onboarding_df['company_tenure_years'].value_counts().sort_index()

## **Statistical Analysis**

In [ ]:
# --- Numerical Features T-tests ---
print("\n--- T-tests for Numerical Features ---")
for col in numerical_columns:
    # Separate data for churned and non-churned employees
    data_stayed = df[df['churn'] == 0][col]
    data_churned = df[df['churn'] == 1][col]

    # Perform independent samples t-test
    # Handle cases where one group might have no variance or insufficient data
    if len(data_stayed) > 1 and data_stayed.std() > 0 and len(data_churned) > 1 and data_churned.std() > 0:
        t_stat, p_value = stats.ttest_ind(data_stayed, data_churned, equal_var=False) # Assuming unequal variances
        print(f"\nFeature: {col}")
        print(f"  T-statistic: {t_stat:.3f}")
        print(f"  P-value: {p_value:.3f}")

        # Interpret the results
        alpha = 0.05
        if p_value < alpha:
            print(f"  Conclusion: Statistically significant difference in {col} between churned and non-churned employees.")
        else:
            print(f"  Conclusion: No statistically significant difference in {col} between churned and non-churned employees.")

        print(f"  Average {col} for Stayed: {data_stayed.mean():.3f}")
        print(f"  Average {col} for Churned: {data_churned.mean():.3f}")
    else:
        print(f"\nFeature: {col}")
        print("  Skipping T-test due to insufficient variance or data in one of the groups.")

# --- Categorical Features Chi-square Tests ---
print("\n--- Chi-square Tests for Categorical Features ---")
for col in categorical_columns:
    print(f"\nFeature: {col}")
    contingency_table = pd.crosstab(df[col], df['churn'])

    # Perform Chi-square test if there are enough observations
    if contingency_table.min().min() > 0 and contingency_table.shape[0] > 1 and contingency_table.shape[1] > 1:
        chi2, p_value, _, _ = stats.chi2_contingency(contingency_table)
        print(f"  Chi-square statistic: {chi2:.3f}")
        print(f"  P-value: {p_value:.3f}")

        alpha = 0.05
        if p_value < alpha:
            print(f"  Conclusion: Statistically significant association between {col} and churn.")
        else:
            print(f"  Conclusion: No statistically significant association between {col} and churn.")
    else:
        print("  Skipping Chi-square test due to insufficient observations in contingency table or too few categories.")

    print("  Contingency Table:")
    print(contingency_table)

In [ ]:
# Initialize a list to store the summary results
summary_data = []

# --- Numerical Features T-tests ---
for col in numerical_columns:
    data_stayed = df[df['churn'] == 0][col]
    data_churned = df[df['churn'] == 1][col]

    if len(data_stayed) > 1 and data_stayed.std() > 0 and len(data_churned) > 1 and data_churned.std() > 0:
        t_stat, p_value = stats.ttest_ind(data_stayed, data_churned, equal_var=False)
        conclusion = "Statistically significant difference" if p_value < 0.05 else "No statistically significant difference"
        summary_data.append({
            'Feature': col,
            'Test': 'Independent Samples T-test',
            'P-value': p_value,
            'Conclusion': conclusion
        })
    else:
        summary_data.append({
            'Feature': col,
            'Test': 'Independent Samples T-test',
            'P-value': 'N/A',
            'Conclusion': 'Skipped (insufficient variance or data)'
        })

# --- Categorical Features Chi-square Tests ---
for col in categorical_columns:
    contingency_table = pd.crosstab(df[col], df['churn'])
    if contingency_table.min().min() > 0 and contingency_table.shape[0] > 1 and contingency_table.shape[1] > 1:
        chi2, p_value, _, _ = stats.chi2_contingency(contingency_table)
        conclusion = "Statistically significant association" if p_value < 0.05 else "No statistically significant association"
        summary_data.append({
            'Feature': col,
            'Test': 'Chi-square Test of Independence',
            'P-value': p_value,
            'Conclusion': conclusion
        })
    else:
        summary_data.append({
            'Feature': col,
            'Test': 'Chi-square Test of Independence',
            'P-value': 'N/A',
            'Conclusion': 'Skipped (insufficient observations or categories)'
        })

# Create DataFrame
summary_df = pd.DataFrame(summary_data)
display(summary_df.round(3))

In [ ]:
churned_df = df[df['churn'] == 1]
stayed_df  = df[df['churn'] == 0]

In [ ]:
churned_df.describe()

Let's compare the mean and std of the employees who stayed and left
- 'target_achievement': Employees who stayed have higher target achievement
- 'working_hours_per_week': Employees who stayed work 3 hours less than the employees who left
- 'job_satisfaction': Employees who stayed is more satisfied with the job
- 'manager_support_score': Employees who stayed got higher support from the manager
- 'company_tenure_years': Employees who stayed tend to have a slightly longer tenure with the company
- 'distance_to_office_km': Employees who stayed live closer to the office, on average

In [ ]:
stayed_df.describe()

## **Multivariate Analysis**

### **Churn Rate**

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(categorical_columns):
  cr = df.groupby(col)['churn'].mean().sort_values(ascending=False)
  cr.plot(
      kind='bar',
      ax=axes[i],
      # color=sns.color_palette('viridis', len(cr)),
      # edgecolor='black',
      width=0.8
  )
  axes[i].set_title(f'Churn Rate by {col}')
  axes[i].set_xlabel('')
  axes[i].set_ylabel('Churn Rate')
  axes[i].tick_params(axis='x', rotation=0)
  for bar in axes[i].patches:
    axes[i].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{bar.get_height():.1%}',
        ha='center',
        va='bottom',
        fontsize=8
    )
for j in range(i+1, len(axes)):
  axes[j].set_visible(False)
plt.suptitle(
    'Churn Rate by Categorical Features',
    fontsize=14,
    fontweight='bold',
    y=1.01
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
for col in categorical_columns:
    print(f"\n--- Churn Counts by {col} ---")
    churn_by_category = df.groupby([col, 'churn']).size().unstack(fill_value=0)
    # Calculate percentage for better understanding
    churn_by_category['Total'] = churn_by_category.sum(axis=1)
    churn_by_category['Churn_Percentage'] = (churn_by_category[1] / churn_by_category['Total']) * 100
    display(churn_by_category.round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10)) # Increased height for better visibility

axes = axes.flatten()

# Plot for work_location
sns.countplot(data=df, x='work_location', hue='churn', ax=axes[0])
axes[0].set_title('Work Location Distribution by Churn Status')
axes[0].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

# Plot for marital_status
sns.countplot(data=df, x='marital_status', hue='churn', ax=axes[1])
axes[1].set_title('Marital Status Distribution by Churn Status')
axes[1].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

# Plot for gender
sns.countplot(data=df, x='gender', hue='churn', ax=axes[2])
axes[2].set_title('Gender Distribution by Churn Status')
axes[2].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

# Plot for education
sns.countplot(data=df, x='education', hue='churn', ax=axes[3])
axes[3].set_title('Education Distribution by Churn Status')
axes[3].tick_params(axis='x', rotation=45)
axes[3].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

# Plot for churn_period
sns.countplot(data=df, x='churn_period', hue='churn', ax=axes[4])
axes[4].set_title('Churn Period Distribution by Churn Status')
axes[4].tick_params(axis='x', rotation=45)
axes[4].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

# Hide any unused subplots (only axes[5] will be hidden now)
for j in range(5, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'Categorical Feature Distributions by Churn Status',
    fontsize=16,
    fontweight='bold',
    y=1.02 # Adjust suptitle position
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust rect for suptitle to prevent overlap
plt.show()

In [ ]:
def plot_stacked_100_by_churn(df, features, num_cols=2):
    num_rows = math.ceil(len(features) / num_cols)
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(6 * num_cols, 5 * num_rows))
    axes = axes.flatten()

    # Ambil warna dari palette yang sudah Anda set (Biru & Oren)
    colors = [sns.color_palette()[0], sns.color_palette()[1]]

    for i, col in enumerate(features):
        ax = axes[i]

        # 1. Buat tabel silang (Crosstab)
        cross_tab = pd.crosstab(df[col], df['churn'])

        # 2. Ubah jadi persentase (Normalisasi baris)
        cross_tab_prop = cross_tab.div(cross_tab.sum(1), axis=0) * 100

        # 3. Plot menggunakan pandas bar plot (stacked=True)
        cross_tab_prop.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.8)

        ax.set_title(f'Churn Percentage by {col}', fontweight='bold')
        ax.set_ylabel('Percentage (%)')
        ax.set_xlabel(col)
        ax.tick_params(axis='x', rotation=0)
        ax.set_ylim(0, 120) # Beri ruang untuk legend di atas

        # 4. Tambahkan label teks persentase di tengah bar
        for n, x in enumerate([*cross_tab_prop.index.values]):
            for (proportion, y_loc) in zip(cross_tab_prop.loc[x],
                                          cross_tab_prop.loc[x].cumsum()):
                if proportion > 5: # Hanya tampilkan jika bar cukup besar
                    ax.text(x=n,
                            y=(y_loc - proportion / 2),
                            s=f'{np.round(proportion, 1)}%',
                            color="white",
                            fontsize=9,
                            fontweight="bold",
                            ha="center")

        ax.legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'], loc='upper center', ncol=2)

    # Hapus subplot kosong
    for j in range(len(features), len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout(rect=[0, 0.03, 1, 0.95], h_pad=3.0)
    plt.show()

# Cara panggil untuk kolom kategorikal Anda:
plot_stacked_100_by_churn(df, categorical_features, num_cols=2)

## **Churned Employees Analysis**

In [ ]:
df_churned = df[df['churn'] == 1]

### **Demographics of Churned Employees**

In [ ]:
# Menggunakan ukuran fig yang lebih efisien dan warna konsisten
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Gender
sns.countplot(data=df_churned, x='gender', ax=axes[0])
axes[0].set_title('Gender of Churned Employees', fontweight='bold')
axes[0].set_xlabel('') # Hilangkan label sumbu X jika judul sudah jelas

# Plot 2: Education
sns.countplot(data=df_churned, x='education', ax=axes[1])
axes[1].set_title('Education of Churned Employees', fontweight='bold')
axes[1].tick_params(axis='x', rotation=0) # Buat horizontal agar rapi
axes[1].set_xlabel('')

# Plot 3: Marital Status
sns.countplot(data=df_churned, x='marital_status', ax=axes[2])
axes[2].set_title('Marital Status of Churned Employees', fontweight='bold')
axes[2].set_xlabel('')

# Menghilangkan garis tepi (spines) agar lebih modern (Opsional)
for ax in axes:
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

### **Job-Related Factors of Churned Employees**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.histplot(data=df_churned, x='experience_years', kde=True, ax=axes[0])
axes[0].set_title('Experience Years of Churned Employees', fontweight='bold')

sns.countplot(data=df_churned, x='work_location', ax=axes[1])
axes[1].set_title('Work Location of Churned Employees', fontweight='bold')

sns.histplot(data=df_churned, x='company_tenure_years', kde=True, ax=axes[2])
axes[2].set_title('Company Tenure Years of Churned Employees', fontweight='bold')

plt.tight_layout()
plt.show()

### **Performance and Financial Data of Churned Employees**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 10)) # Increased height for better visualization
sns.histplot(data=df_churned, x='salary', kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Salary Distribution of Churned Employees', fontweight='bold')

sns.histplot(data=df_churned, x='target_achievement', kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Target Achievement Distribution of Churned Employees', fontweight='bold')

sns.histplot(data=df_churned, x='working_hours_per_week', kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Working Hours Per Week of Churned Employees', fontweight='bold')

sns.histplot(data=df_churned, x='commission_rate', kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Commission Rate of Churned Employees', fontweight='bold')

plt.tight_layout()
plt.show()

### **Job Satisfaction and Manager Support for Churned Employees**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot for Distance to Office
sns.countplot(data=df_churned, x='distance_to_office_km', hue='distance_to_office_km', ax=axes[0], legend=False)
axes[0].set_title('Distance to Office of Churned Employees')
axes[0].set_xlabel('Distance to Office (km)')
axes[0].set_ylabel('Number of Churned Employees')
total_churned = len(df_churned)
for p in axes[0].patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total_churned)
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    axes[0].annotate(percentage, (x, y), ha='center', va='bottom')

# Plot for Churn Period
sns.countplot(data=df_churned, x='churn_period', hue='churn_period', ax=axes[1], legend=False)
axes[1].set_title('Churn Period of Churned Employees')
axes[1].set_xlabel('Churn Period')
axes[1].set_ylabel('Number of Churned Employees')
for p in axes[1].patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total_churned)
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    axes[1].annotate(percentage, (x, y), ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Pastikan data diurutkan berdasarkan jarak (bukan berdasarkan jumlah churn)
all_dist = df_churned['distance_to_office_km'].value_counts().sort_index().reset_index()
all_dist.columns = ['distance', 'count']

# 2. Perlebar ukuran figure agar batang tidak berhimpitan
plt.figure(figsize=(20, 7))
base_color = sns.color_palette()[0]

# 3. Plot menggunakan barplot
ax = sns.barplot(data=all_dist, x='distance', y='count', color=base_color)

# 4. Tambahkan Judul & Label
plt.title('Distance to Office of Churned Employees', fontsize=18, fontweight='bold', pad=25)
plt.xlabel('Distance to Office (km)', fontsize=12)
plt.ylabel('Number of Churned Employees', fontsize=12)

# 5. Tambahkan Label Persentase dengan ukuran kecil agar tidak overlap
total_churned = len(df_churned)
for p in ax.patches:
    if p.get_height() > 0:
        percentage = f'{100 * p.get_height() / total_churned:.1f}%'
        ax.annotate(percentage,
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='baseline',
                    fontsize=8, # Ukuran font diperkecil
                    fontweight='bold',
                    xytext=(0, 5),
                    textcoords='offset points')

# Menampilkan label sumbu X setiap 2 atau 5 angka jika terlalu padat (opsional)
# plt.xticks(rotation=0)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# 1. Ambil data Top 10 Jarak agar tidak terlalu padat
top_10_dist = df_churned['distance_to_office_km'].value_counts().head(10).reset_index()
top_10_dist.columns = ['distance', 'count']
# Urutkan berdasarkan jarak (km) agar sumbu X logis
top_10_dist = top_10_dist.sort_values(by='distance')

# 2. Mulai Plotting
plt.figure(figsize=(12, 6))
base_color = sns.color_palette()[0] # Menggunakan warna pertama dari palette global

ax = sns.barplot(data=top_10_dist, x='distance', y='count', color=base_color)

# 3. Percantik Tampilan (Sesuai prinsip "Clean Visualization")
plt.title('Top 10 Distance to Office of Churned Employees', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Distance to Office (km)', fontsize=12)
plt.ylabel('Number of Churned Employees', fontsize=12)

# 4. Tambahkan Label Persentase yang Rapi
total_churned = len(df_churned)
for p in ax.patches:
    if p.get_height() > 0:
        percentage = f'{100 * p.get_height() / total_churned:.1f}%'
        ax.annotate(percentage,
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='baseline',
                    fontsize=11, fontweight='bold',
                    xytext=(0, 8), # Memberi jarak antara angka dan bar
                    textcoords='offset points')

# Menghapus garis tepi agar lebih modern
sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
order_period = ['Onboarding', '1 Month', '3 Months']

# Menggunakan warna yang sama untuk konsistensi
sns.countplot(data=df_churned, x='churn_period', order=order_period, color=base_color)

plt.title('Churn Period of Churned Employees', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Churn Period', fontsize=12)
plt.ylabel('Number of Churned Employees', fontsize=12)

# Label Persentase
for p in plt.gca().patches:
    percentage = f'{100 * p.get_height() / total_churned:.1f}%'
    plt.gca().annotate(percentage, (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=9, fontweight='bold',
                xytext=(0, 8), textcoords='offset points')

sns.despine()
plt.tight_layout()
plt.show()

## **Correlation**

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_corr = df.copy()

for c in df_corr.select_dtypes('object').columns:
  df_corr[c] = LabelEncoder().fit_transform(df_corr[c])
corr = df_corr.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8}
)
ax.set_title(
    'Feature Correlation Heatmap',
    fontsize=14,
    fontweight='bold'
)
plt.tight_layout()
plt.show()

# **Data Prepocessing**

## **Feature Selection**

### **Feature Removal**

In [ ]:
# drop feature
df = df.drop(['employee_id', 'churn_period'], axis=1)

In [ ]:
df.info()

### **Feature Extraction**

#### **Overtime Ratio**

In [ ]:
# It could highlight employees with significant overtime relative to their standard hours

df['overtime_ratio'] = (
    df['overtime_hours_per_week'] /
    df['working_hours_per_week']
)

####**Overall Satisfaction**

In [ ]:
# Interaction terms: job_satisfaction & manager_support_score
# It might reveal if high job satisfaction only leads to retention when manager support is also high.
# It might boost performance or help the model find that relationship more easily

df['overall_satisfaction'] = df['job_satisfaction'] * df['manager_support_score']

#### **Unachieved Target**

In [ ]:
# It shows the target that haven't been achieved within a month

df['unachieved_target'] = (
    df['monthly_target'] *
    (1 - df['target_achievement'])
)

#### **Salary Level**

In [ ]:
#df['salary_level'] = pd.qcut(
 #   df['salary'],
  #  q=3,
   # labels=['Low','Medium','High']
#)

#### **Distance Group**

In [ ]:
# It might capture thresholds beyond which distance becomes a significant churn factor

df['distance_group'] = pd.cut(
    df['distance_to_office_km'],
    bins=[0,10,25,50],
    labels=['Near','Medium','Far']
)

## **Define Columns**

In [ ]:
numerical_columns = df.select_dtypes(include=[np.number]).columns
numerical_columns = numerical_columns.drop(['churn'])

categorical_columns = df.select_dtypes(exclude=[np.number]).columns

print("Numerical Columns:")
print(numerical_columns)
print("\nCategorical Columns:")
print(categorical_columns)

## **Train-Test Split**

In [ ]:
X = df.drop('churn', axis=1)
y = df['churn']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## **Pipeline**

### **Preprocessing**

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_columns),
    ('cat', cat_pipeline, categorical_columns)
])

### **Feature Selection**

In [ ]:
# Find best k feature
X_encoded = preprocessor.fit_transform(X_train)

k_values = range(5, 31, 5)

scores = []

for k in k_values:
    selector = SelectKBest(
        score_func=f_classif,
        k=k
    )

    X_selected = selector.fit_transform(
        X_encoded,
        y_train
    )

    model = LogisticRegression(
        max_iter=1000
    )

    score = cross_val_score(
        model,
        X_selected,
        y_train,
        cv=5,
        scoring='f1'
    ).mean()

    scores.append(score)

In [ ]:
plt.plot(
    k_values,
    scores,
    marker='o'
)

plt.xlabel("k features")
plt.ylabel("F1 Score")

plt.title("Finding Best k")

plt.show()

In [ ]:
best_k = k_values[scores.index(max(scores))]

print("Best k:", best_k)

In [ ]:
selector = SelectKBest(
    score_func=f_classif,
    k='all'
)

selector.fit(X_encoded, y_train)

# Get feature names after one-hot encoding
# The preprocessor creates a sparse matrix, so we need to get the feature names correctly
feature_names = preprocessor.get_feature_names_out()

feature_scores = pd.DataFrame({
    'Feature': feature_names,
    'Score': selector.scores_
})

feature_scores = feature_scores.sort_values(
    by='Score',
    ascending=False
)

print(feature_scores)

In [ ]:
feature_selector = SelectKBest(
    score_func=f_classif,
    k=best_k
)

### **Oversampling (SMOTE)**

In [ ]:
smote = SMOTE(random_state=42)

# **Modeling Pipeline**

## **Pipeline Template**

In [ ]:
def create_pipeline(model):
  pipeline = ImbPipeline([
      ('preprocessor', preprocessor),
      ('feature_selector', feature_selector),
      ('smote', smote),
      ('model', model)
  ])

  return pipeline

## **Model 1 - Logistic Regression**

In [ ]:
lr_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    ('smote', smote),
    ('model', LogisticRegression(
        max_iter=1000
    ))
])

lr_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = lr_pipeline.predict(X_test)


print(classification_report(y_test, y_pred))

## **Model 2 - Decision Tree**

In [ ]:
dt_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    ('smote', smote),
    ('model', DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])

dt_pipeline.fit(X_train, y_train)

## **Model 3 - Random Forest**

In [ ]:
rf_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    ('smote', smote),
    ('model', RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

## **Model 4 - Gradient Boosting**

In [ ]:
gb_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    ('smote', smote),
    ('model', GradientBoostingClassifier())
])

gb_pipeline.fit(X_train, y_train)

## **Model 5 - XGBoost**

In [ ]:
xgb_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    ('smote', smote),
    ('model', XGBClassifier(
        eval_metric='logloss'
    ))
])

xgb_pipeline.fit(X_train, y_train)